# UdaPlay Part 2

A stateful research agent that retrieves local game data, evaluates its sufficiency, searches the web when required, and produces cited reports.

## Environment and database

Run Part 1 first. This notebook uses its `chromadb` database and `udaplay` collection with the same embedding configuration.

In [ ]:
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [ ]:
import json

from lib.udaplay import (
    configure_environment, ResearchAgent, EvaluationReport, ResearchReport,
    retrieve_game, evaluate_retrieval, game_web_search,
)

settings = configure_environment()
agent = ResearchAgent.from_environment(database_path="chromadb")
assert agent.store.collection.count() == 15
print("Research model:", settings["model"])
print("Database loaded:", agent.store.collection.name)

## Tools and state machine

- `retrieve_game`: query the persistent vector database and retain source metadata.
- `evaluate_retrieval`: return a typed `EvaluationReport` with sufficiency, confidence and an explanation.
- `game_web_search`: use Tavily when evidence is absent or insufficient.

The implementations are in `lib/udaplay.py`. The supplied `lib/state_machine.py` executes explicit retrieve → evaluate → optional web search → report nodes. Local support below 70% routes to web search even if the evaluator marks it useful. Retrieved text is treated as data rather than instructions. Reports may cite only supplied source IDs.

In [ ]:
print("Registered stages:", list(agent.workflow.steps))
for function in (retrieve_game, evaluate_retrieval, game_web_search):
    print(function.__name__ + ": " + function.__doc__)

## Required example queries

The first two questions concern the local dataset. Mortal Kombat X is absent, so it must demonstrate web fallback. Each report shows stage activity, the evaluation explanation, the answer, confidence, and citations. These are concise decision summaries, not hidden model reasoning.

In [ ]:
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released on PlayStation 5?",
]
reports = []
for question in questions:
    report = agent.invoke(question, session_id="examples")
    reports.append(report)
    report.display()
    print("\nStructured report:")
    print(report.model_dump_json(indent=2))

## Conversation memory

A follow-up resolves its reference from the same session. Other sessions remain isolated. Memory lasts for the current Python process; persistent long-term memory is an optional extension.

In [ ]:
followup = agent.invoke("Which company published that game?", session_id="examples")
followup.display()
print("Session turns:", len(agent.history["examples"]))
assert len(agent.history["examples"]) == 4

## Performance and evidence checks

The assertions below verify demonstrated behavior, rather than assuming the prompts caused the intended route. Confidence is the model’s self-assessed support, not a calibrated probability of correctness. Web results may change over time, and the fifteen-record dataset is intentionally small.

In [ ]:
summary = [{"question": report.question, "route": report.route,
            "confidence": report.confidence, "sources": len(report.sources),
            "stages": [event["stage"] for event in report.trace]}
           for report in reports]
print(json.dumps(summary, indent=2, ensure_ascii=False))
assert any(report.route == "local" for report in reports)
assert reports[2].route == "web", "Mortal Kombat X must demonstrate web fallback"
for report in reports:
    assert report.sources, "Each example needs supporting citations"
    assert report.trace[0]["stage"] == "retrieve_game"
    assert report.trace[1]["stage"] == "evaluate_retrieval"
    if report.route == "web":
        assert all(source.url.startswith(("https://", "http://")) for source in report.sources)
print("Required local, fallback, trace and citation demonstrations passed.")